# 02 - Auditoria dos dados brutos
Objetivo: medir quantos registros cada tipo de problema afeta, por arquivo e por coluna. Esses números são o "antes" que comprova a limpeza e formam o dicionário de inconsistências do README.


In [1]:
import pandas as pd

compras = pd.read_csv("../data/raw/compras.csv", dtype=str)
itens = pd.read_csv("../data/raw/itens.csv", dtype=str)
resultados = pd.read_csv("../data/raw/resultados.csv", dtype=str)


---
## 1. Documentos (CNPJ/CPF)


In [2]:
def classificar_documento(doc, tipo):
    if pd.isna(doc):
        return "vazio"
    if tipo == "PE":
        return "estrangeiro"
    tamanho = 14 if tipo == "PJ" else 11
    if doc != doc.strip():
        return "espaços extras"
    if not doc.isdigit():
        return "máscara"
    if len(doc) < tamanho:
        return "zeros perdidos"
    if len(doc) == tamanho:
        return "limpo"
    return "dígitos demais"


In [3]:
compras["cnpj_orgao"].apply(lambda doc: classificar_documento(doc, "PJ")).value_counts()


cnpj_orgao
limpo             1658
máscara            176
espaços extras     131
zeros perdidos      77
Name: count, dtype: int64

In [4]:
situacao_ni = [classificar_documento(doc, tipo)
               for doc, tipo in zip(resultados["ni_fornecedor"], resultados["tipo_pessoa"])]

pd.crosstab(pd.Series(situacao_ni, name="situação"), resultados["tipo_pessoa"], margins=True)


tipo_pessoa,PE,PF,PJ,All
situação,,,,
espaços extras,0,293,695,988
estrangeiro,21,0,0,21
limpo,0,3687,9216,12903
máscara,0,392,1239,1631
zeros perdidos,0,159,194,353
All,21,4531,11344,15896
